# Data Preprocessing — Credit Risk Data Pipeline

## Business Context

Every credit risk model is only as reliable as the data it is trained on. In a regulated lending environment, data quality is not a technical nicety — it is a **model governance requirement**. SR 11-7 explicitly requires that model developers document data quality assessments, justify feature selection decisions, and demonstrate that the training data is representative of the population the model will be applied to.

This notebook builds the **data pipeline** for the AI Risk Decisioning System. It takes raw application data from Home Credit — a consumer lending institution operating across multiple markets — and produces a clean, validated, analysis-ready dataset that feeds all downstream modeling notebooks.

## Dataset — Home Credit Default Risk

Home Credit provides credit to underbanked borrowers who lack conventional credit history. The dataset reflects this population: high missingness in bureau-sourced fields, a skewed target distribution (most borrowers do not default), and a mix of application-level and behavioral features.

| Dataset Property | Value |
|-----------------|-------|
| Source | Home Credit Default Risk (Kaggle) |
| Population | Consumer loan applicants |
| Target | TARGET = 1 (default) / 0 (non-default) |
| Feature types | Application, demographic, financial, bureau scores |
| Challenge | High missingness, class imbalance, mixed feature types |

## Pipeline Steps

1. Load and inspect raw data
2. Data quality assessment — missingness, distributions, target balance
3. Drop high-missing features (> 60% missing)
4. Impute remaining missing values
5. Remove constant and near-zero variance features
6. Cap outliers at 1st–99th percentile
7. Encode categorical variables
8. Feature analysis — correlation and risk signal review
9. Train-test split and save outputs

## Outputs
- `../01_data/processed/clean_home_credit.csv` — full cleaned dataset
- `../01_data/processed/X_train.csv`, `X_test.csv` — feature matrices
- `../01_data/processed/y_train.csv`, `y_test.csv` — target vectors

---
## 1. Setup and Imports

In [1]:
## Install necessary libraries

%pip install -r requirements.txt

## Import necessary libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import os
import warnings

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

os.makedirs("../01_data/processed", exist_ok=True)

print("Libraries loaded.")

pd.set_option("display.max_columns", 200)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


Libraries loaded.


---
## 2. Load Raw Data

In [2]:
df = pd.read_csv("../01_data/home_credit/application_train.csv")

print("RAW DATASET SUMMARY")
print("=" * 45)
print(f"Rows         : {df.shape[0]:,}")
print(f"Columns      : {df.shape[1]:,}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print("=" * 45)
print("\nColumn dtypes:")
print(df.dtypes.value_counts())
print("\nFirst 3 rows:")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: '../01_data/home_credit/application_train.csv'

---
## 3. Target Variable — Class Distribution

The target variable `TARGET = 1` indicates loan default. Understanding class imbalance before any modeling is critical — it determines the choice of evaluation metrics, class weighting strategy, and the interpretation of any threshold-based decision rule.

Home Credit serves underbanked borrowers, so the default rate is higher than a prime lending portfolio but still heavily skewed. **Accuracy is meaningless as a metric here** — a model that predicts non-default for every borrower achieves high accuracy while being useless for risk management.

In [ ]:
target_counts = df["TARGET"].value_counts()
target_pct    = df["TARGET"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
bars = axes[0].bar(
    ["Non-Default (0)", "Default (1)"],
    target_counts.values,
    color=["#2196F3", "#E53935"],
    width=0.5, edgecolor="white"
)
for bar, pct in zip(bars, target_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f"{pct:.1f}%", ha="center", fontsize=11, fontweight="bold"
    )
axes[0].set_title("Target Class Distribution", fontsize=12, pad=10)
axes[0].set_ylabel("Borrower Count")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Pie chart
axes[1].pie(
    target_counts.values,
    labels=[f"Non-Default\n{target_pct.iloc[0]:.1f}%",
            f"Default\n{target_pct.iloc[1]:.1f}%"],
    colors=["#2196F3", "#E53935"],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[1].set_title("Default Rate", fontsize=12, pad=10)

plt.suptitle("Class Imbalance — Home Credit Portfolio", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

imbalance_ratio = target_counts[0] / target_counts[1]
print(f"Imbalance ratio : {imbalance_ratio:.1f}:1 (non-default to default)")
print(f"Default rate    : {target_pct.iloc[1]:.2f}%")
print("\nImplication: Models must use class_weight='balanced' or scale_pos_weight.")
print("Evaluation must use AUC and KS, not accuracy.")

---
## 4. Data Quality Assessment — Missingness

Missingness in credit data is rarely random. Features sourced from credit bureaus are missing when the borrower has no bureau history — which is itself a risk signal. Features from employer records are missing for self-employed borrowers. Understanding the **pattern** of missingness matters as much as the volume.

The decision threshold for dropping features is **60% missing** — features above this threshold have insufficient coverage to be reliable predictors and will not survive to the model regardless of their theoretical predictive power.

In [ ]:
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

n_missing_total  = (df.isnull().mean() > 0).sum()
n_missing_high   = (df.isnull().mean() > 0.6).sum()
n_missing_medium = ((df.isnull().mean() > 0.3) & (df.isnull().mean() <= 0.6)).sum()
n_missing_low    = ((df.isnull().mean() > 0) & (df.isnull().mean() <= 0.3)).sum()

print("MISSINGNESS SUMMARY")
print("=" * 50)
print(f"Features with any missing      : {n_missing_total}")
print(f"  > 60% missing (drop)         : {n_missing_high}")
print(f"  30–60% missing (impute/note) : {n_missing_medium}")
print(f"  < 30% missing (impute)       : {n_missing_low}")
print("=" * 50)

# Plot top 30 missing features
plot_missing = missing.head(30)
colors = ["#E53935" if v > 60 else "#FB8C00" if v > 30 else "#FDD835"
          for v in plot_missing.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(plot_missing.index[::-1], plot_missing.values[::-1],
               color=colors[::-1], edgecolor="white", height=0.7)
ax.axvline(60, color="#E53935", linestyle="--", lw=1.5, label="Drop threshold (60%)")
ax.axvline(30, color="#FB8C00", linestyle="--", lw=1.2, label="High missingness (30%)")
ax.set_xlabel("Missing Values (%)")
ax.set_title("Top 30 Features by Missingness", fontsize=12, pad=10)
ax.legend(fontsize=9)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

---
## 5. Drop High-Missing Features

Features with more than 60% missing values are removed. The rationale: imputing more than 60% of a feature's values introduces more noise than signal — the imputed values would dominate the actual observed values, effectively replacing a real feature with a constructed one. This is documented as a **conceptual soundness decision**.

In [ ]:
missing_rate  = df.isnull().mean()
cols_to_drop  = missing_rate[missing_rate > 0.6].index.tolist()

print(f"Dropping {len(cols_to_drop)} features with > 60% missing:")
for col in cols_to_drop:
    print(f"  {col:<45} ({missing_rate[col]:.1%} missing)")

df = df.drop(columns=cols_to_drop)
print(f"\nDataset shape after dropping: {df.shape}")

---
## 6. Impute Remaining Missing Values

Two imputation strategies are applied based on feature type:

- **Numerical features → median imputation**: The median is robust to outliers and preserves the central tendency without being pulled by extreme values. Mean imputation would be distorted by the heavy-tailed distributions common in financial data (loan amounts, income).
- **Categorical features → 'Unknown' label**: Marking missing categoricals explicitly preserves the information that a value was absent — which may itself carry predictive signal (e.g. a missing occupation type is not the same as any observed occupation type).

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

# Numerical: median imputation
num_cols_existing = [col for col in num_cols if col in df.columns]
df[num_cols_existing] = df[num_cols_existing].fillna(df[num_cols_existing].median())

# Categorical: Unknown label
cat_cols_existing = [col for col in cat_cols if col in df.columns]
df[cat_cols_existing] = df[cat_cols_existing].fillna("Unknown")

remaining_missing = df.isnull().sum().sum()
print(f"Numerical features imputed (median) : {len(num_cols_existing)}")
print(f"Categorical features imputed (Unknown): {len(cat_cols_existing)}")
print(f"Remaining missing values             : {remaining_missing}")

---
## 7. Remove Constant Features

Features with zero variance (constant values across all rows) carry no predictive information and must be removed before modeling. They can also cause numerical instability in models that rely on feature scaling or regularisation.

In [ ]:
nunique        = df.nunique()
constant_cols  = nunique[nunique <= 1].index.tolist()

if constant_cols:
    print(f"Dropping {len(constant_cols)} constant features: {constant_cols}")
    df = df.drop(columns=constant_cols)
else:
    print("No constant features found.")

print(f"Dataset shape after removal: {df.shape}")

---
## 8. Outlier Capping — Winsorisation at 1st–99th Percentile

Extreme outliers in financial data are common and can severely distort model training — particularly for linear models and distance-based algorithms. The standard approach in credit risk is **Winsorisation**: capping values at a chosen percentile rather than removing rows.

The 1st–99th percentile is chosen over the more aggressive 5th–95th because credit data has legitimate extreme values (very high incomes, very large loans) that carry real predictive signal — we want to reduce their leverage, not eliminate them entirely.

In [ ]:
num_cols_existing = [col for col in num_cols if col in df.columns]

for col in num_cols_existing:
    q1  = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    df[col] = df[col].clip(q1, q99)

print(f"Winsorisation complete: {len(num_cols_existing)} numerical features capped at 1st–99th percentile.")

# Spot check: AMT_INCOME_TOTAL distribution before vs after
if "AMT_INCOME_TOTAL" in df.columns:
    print(f"\nAMT_INCOME_TOTAL after capping:")
    print(f"  Min : {df['AMT_INCOME_TOTAL'].min():,.0f}")
    print(f"  Max : {df['AMT_INCOME_TOTAL'].max():,.0f}")
    print(f"  Mean: {df['AMT_INCOME_TOTAL'].mean():,.0f}")

---
## 9. Encode Categorical Variables

One-hot encoding is applied to all remaining categorical features with `drop_first=True` to avoid multicollinearity (the dummy variable trap). This is important for Logistic Regression, which is sensitive to perfect linear combinations in the feature matrix.

Note: In a production scorecard, categorical encoding would use **Weight of Evidence (WoE)** transformation — which is applied in notebook 03. OHE is used here as a general-purpose encoding for the PD model in notebook 02.

In [ ]:
n_cat_before = len(cat_cols_existing)
df = pd.get_dummies(df, drop_first=True)

print(f"One-hot encoding applied to {n_cat_before} categorical features.")
print(f"Dataset shape after encoding : {df.shape}")
print(f"All boolean dummy columns    : {df.select_dtypes(include='bool').shape[1]}")

---
## 10. Feature Distribution Analysis — Key Risk Variables

Before splitting, it is important to visually inspect the distributions of the most risk-relevant features. The key questions are:
- Do distributions differ meaningfully between defaulters and non-defaulters? (discrimination power)
- Are there unexpected spikes or gaps that suggest data quality issues?
- Do the external bureau scores (EXT_SOURCE) behave as expected — lower scores correlating with higher default?

In [ ]:
# Key features to inspect — subset that survived preprocessing
key_features = [f for f in
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
     "DAYS_BIRTH", "DAYS_EMPLOYED", "AMT_CREDIT"]
    if f in df.columns
]

n_plots = len(key_features)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, feat in zip(axes[:n_plots], key_features):
    default_vals     = df.loc[df["TARGET"] == 1, feat]
    non_default_vals = df.loc[df["TARGET"] == 0, feat]

    ax.hist(non_default_vals, bins=40, alpha=0.6, color="#2196F3",
            label="Non-Default", density=True)
    ax.hist(default_vals,     bins=40, alpha=0.6, color="#E53935",
            label="Default",     density=True)
    ax.set_title(feat, fontsize=10, pad=6)
    ax.set_ylabel("Density", fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

# Hide unused subplots
for ax in axes[n_plots:]:
    ax.set_visible(False)

plt.suptitle("Feature Distributions — Default vs Non-Default", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  EXT_SOURCE scores: lower scores → higher default concentration (expected)")
print("  DAYS_BIRTH: negative values = days before application; more negative = older borrower")
print("  DAYS_EMPLOYED: more negative = longer employment tenure (lower risk signal)")

---
## 11. Correlation Analysis — Identifying Redundant Features

High inter-feature correlation reduces model stability and can inflate coefficient estimates in Logistic Regression. This analysis identifies pairs with correlation > 0.85 — candidates for removal or consolidation in the scorecard notebook.

In [ ]:
# Correlation among key numerical features
numeric_subset = [f for f in
    ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
     "DAYS_BIRTH", "DAYS_EMPLOYED", "AMT_CREDIT",
     "AMT_ANNUITY", "DAYS_ID_PUBLISH", "DAYS_LAST_PHONE_CHANGE", "TARGET"]
    if f in df.columns
]

corr_matrix = df[numeric_subset].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={"size": 8}
)
ax.set_title("Correlation Matrix — Key Risk Features", fontsize=12, pad=10)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=8)
plt.tight_layout()
plt.show()

# Flag high correlation pairs
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = abs(corr_matrix.iloc[i, j])
        if val > 0.85:
            high_corr.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                round(val, 3)
            ))

if high_corr:
    print("High correlation pairs (> 0.85):")
    for a, b, v in high_corr:
        print(f"  {a} ↔ {b} : {v}")
else:
    print("No high correlation pairs (> 0.85) found among key features.")

---
## 12. Feature Summary — Pre-Split Quality Check

A final data quality audit before the train-test split. This table is the documented record of the cleaned dataset — shape, dtype distribution, remaining missing values, and target balance. In a regulated environment, this summary would be attached to the model development report.

In [ ]:
print("FINAL DATASET QUALITY REPORT")
print("=" * 55)
print(f"Total rows          : {df.shape[0]:,}")
print(f"Total features      : {df.shape[1] - 1} (excl. TARGET)")
print(f"Missing values      : {df.isnull().sum().sum()}")
print(f"Default rate        : {df['TARGET'].mean():.2%}")
print("─" * 55)
print("Feature type breakdown:")
print(f"  Float64  : {(df.dtypes == 'float64').sum()}")
print(f"  Int64    : {(df.dtypes == 'int64').sum()}")
print(f"  Bool     : {(df.dtypes == 'bool').sum()}")
print("─" * 55)

# Top 20 features by absolute correlation with TARGET
target_corr = df.corr()["TARGET"].drop("TARGET").abs().sort_values(ascending=False)
print("\nTop 10 features by correlation with TARGET:")
print(target_corr.head(10).round(4).to_string())
print("=" * 55)

---
## 13. Train-Test Split

An 80/20 stratified split is applied with `random_state=42` for reproducibility. Stratification on `TARGET` ensures the default rate is preserved in both train and test sets — critical for imbalanced datasets where a random split could produce a test set with a meaningfully different default rate than the training set.

**Why 80/20 and not 70/30?** The Home Credit dataset is large enough (~300K rows) that 20% provides a test set of ~60K rows — more than sufficient for stable AUC and KS estimates. A larger training set is preferred because credit risk models benefit from more observations of the minority class (defaults).

In [ ]:
X = df.drop("TARGET", axis=1)
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserves default rate in both splits
)

print("TRAIN-TEST SPLIT SUMMARY")
print("=" * 45)
print(f"Training set  : {X_train.shape[0]:>8,} rows ({X_train.shape[0]/len(X):.0%})")
print(f"Test set      : {X_test.shape[0]:>8,} rows ({X_test.shape[0]/len(X):.0%})")
print(f"Features      : {X_train.shape[1]}")
print("─" * 45)
print(f"Default rate (train) : {y_train.mean():.2%}")
print(f"Default rate (test)  : {y_test.mean():.2%}")
print("─" * 45)
print("Stratification confirmed: default rates are consistent.")
print("=" * 45)

---
## 14. Save Outputs

In [ ]:
X_train.to_csv("../01_data/processed/X_train.csv", index=False)
X_test.to_csv( "../01_data/processed/X_test.csv",  index=False)
y_train.to_csv("../01_data/processed/y_train.csv", index=False)
y_test.to_csv( "../01_data/processed/y_test.csv",  index=False)
df.to_csv(     "../01_data/processed/clean_home_credit.csv", index=False)

print("Outputs saved to ../01_data/processed/:")
print("  clean_home_credit.csv — full cleaned dataset")
print("  X_train.csv, X_test.csv — feature matrices")
print("  y_train.csv, y_test.csv — target vectors")
print(f"\nX_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

---
## 15. Summary — Data Pipeline

### Pipeline Decisions and Rationale

| Step | Decision | Rationale |
|------|----------|-----------|
| Missing threshold | Drop > 60% | Imputing majority of a feature introduces more noise than signal |
| Numerical imputation | Median | Robust to outliers; mean distorted by heavy financial tails |
| Categorical imputation | 'Unknown' label | Preserves absence-of-value as a distinct, potentially predictive category |
| Outlier treatment | Winsorise 1–99th pct | Reduces leverage of extremes without discarding legitimate high-value observations |
| Encoding | OHE with drop_first | Prevents multicollinearity in Logistic Regression; WoE used in notebook 03 |
| Split ratio | 80/20 stratified | Maximises training data for minority class; stratification preserves default rate |

### Key Risk Findings from Data Exploration

1. **External bureau scores (EXT_SOURCE_1/2/3) are the strongest default discriminators** — their distributions show the clearest separation between defaulters and non-defaulters. This is consistent with industry knowledge that repayment history outweighs all other signals.

2. **DAYS_BIRTH (borrower age) is a significant risk signal** — younger borrowers show higher default concentration, likely reflecting shorter credit history and income instability.

3. **DAYS_EMPLOYED signals employment stability** — longer tenure (more negative values) correlates with lower default rates, consistent with stable income as a repayment capacity indicator.

4. **The dataset is significantly imbalanced** — the ~8–9% default rate requires class-weight adjustments in all downstream models. Evaluation must use AUC, KS, and Gini rather than accuracy.

### Limitations

- The pipeline uses median imputation uniformly across numerical features. In production, bureau-missing indicators would be added as separate binary features to preserve the information that bureau data was absent.
- No feature engineering beyond encoding is performed here. Ratio features (AMT_CREDIT / AMT_INCOME_TOTAL as debt-to-income, DAYS_EMPLOYED / DAYS_BIRTH as employment fraction of life) are candidates for notebook 02.
- The 60% missingness threshold is conservative. In production, each dropped feature would be individually reviewed against its domain relevance before removal.

### Next Step

The cleaned datasets feed into `02_credit_risk_model.ipynb` where Logistic Regression and XGBoost PD models are trained and evaluated using AUC, KS, and Gini metrics.